# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [1]:
import pandas as pd, os

CANDIDATES = ['../06-midterm-walmart/data/product_catalog.csv', 'product_catalog.csv',
              '/content/product_catalog.csv', 'sample_data/product_catalog.csv']
PATH = next((p for p in CANDIDATES if os.path.exists(p)), None)
cat = pd.read_csv(PATH)
print(f"Loaded {len(cat)} rows from {PATH}")
cat.head()

Loaded 45 rows from product_catalog.csv


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [2]:
is_pop = cat['product_name'].str.lower().str.contains('pop')
pop = cat[is_pop]
print(len(pop), "rows look like Pop-Tart products")
pop

4 rows look like Pop-Tart products


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [3]:
is_straw = pop['product_name'].str.lower().str.contains('straw')
strawberry = pop[is_straw]
not_straw = pop[~is_straw]
print(strawberry[['sku', 'product_name', 'category', 'package_size']])
print(len(not_straw), "Pop-Tart rows are not strawberry")

             sku          product_name         category package_size
0          PT-12  pop-tarts strawberry  Breakfast Foods       4-pack
1     SKU#459812   Strawberry Pop-Tart        Breakfast          4pk
2  POPTART-STRAW  Pop-Tarts Strawberry        Breakfast       6 pack
3  PT_STRAWBERRY  pop-tarts strawberry        breakfast       4 pack
0 Pop-Tart rows are not strawberry


**Pop-Tart rows that are a different product:** _None._ All 4 rows matching `pop` (`PT-12`, `SKU#459812`, `POPTART-STRAW`, `PT_STRAWBERRY`) are strawberry, so no other Pop-Tart flavor (frosted s'mores, brown sugar, etc.) exists in this catalog to protect. The filter `contains('pop')` also did not catch false positives such as "Monopoly" (the letters are m-o-n-**o-p-o**-l-y, never "pop"). One caveat: `POPTART-STRAW` is a **6 pack** while the others are 4-packs, so it may be a different pack size. If other flavors were added later, they would need to stay separate.

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [4]:
CANON = 'PT-12'
sku_map = {'PT-12': CANON, 'SKU#459812': CANON, 'POPTART-STRAW': CANON, 'PT_STRAWBERRY': CANON}

cat['canonical_sku'] = cat['sku'].map(sku_map)
print("map alone, nulls:", cat['canonical_sku'].isna().sum())

cat['canonical_sku'] = cat['sku'].map(sku_map).fillna(cat['sku'])
print("with fillna, nulls:", cat['canonical_sku'].isna().sum())

map alone, nulls: 41
with fillna, nulls: 0


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [5]:
print(cat['category'].value_counts().sort_index())
cat['category_clean'] = cat['category'].str.strip().str.title()
print(cat['category_clean'].value_counts().sort_index())



category
ALCOHOL                1
Alcoholic Beverages    1
Baby                   2
Baby Products          1
Breakfast              2
Breakfast Foods        1
ELECTRONICS            2
EMERGENCY              3
ENTERTAINMENT          3
Electronics            1
Emergency              5
Emergency Supplies     3
GROCERY                3
Groceries              1
Grocery                1
HOUSEHOLD              1
Household              1
Snacks                 1
baby                   1
breakfast              1
electronics            1
emergency              1
entertainment          2
grocery                1
health                 2
household              2
snacks                 1
Name: count, dtype: int64
category_clean
Alcohol                1
Alcoholic Beverages    1
Baby                   3
Baby Products          1
Breakfast              3
Breakfast Foods        1
Electronics            4
Emergency              9
Emergency Supplies     3
Entertainment          5
Groceries              1


**Still two labels for one thing after strip + title-case:**

| Label A | Label B | Same category? |
|---|---|---|
| Breakfast | Breakfast Foods | yes |
| Emergency | Emergency Supplies | yes |
| Alcohol | Alcoholic Beverages | yes |
| Grocery | Groceries | yes |
| Baby | Baby Products | yes |

Case/whitespace cleanup only removes spelling noise. These pairs need an explicit synonym map (next cell).

In [6]:
# still two labels for one thing, so map them together
cat_synonyms = {'Breakfast Foods': 'Breakfast', 'Emergency Supplies': 'Emergency',
                'Alcoholic Beverages': 'Alcohol', 'Groceries': 'Grocery', 'Baby Products': 'Baby'}
cat['category_clean'] = cat['category_clean'].replace(cat_synonyms)
print(cat['category_clean'].value_counts())

category_clean
Emergency        12
Grocery           6
Entertainment     5
Breakfast         4
Electronics       4
Baby              4
Household         4
Alcohol           2
Snacks            2
Health            2
Name: count, dtype: int64


### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [7]:
import re

def name_key(s):
    s = re.sub(r'[^a-z0-9\s]', ' ', s.lower())
    return ' '.join(sorted(s.split()))

cat['name_key'] = cat['product_name'].map(name_key)
groups = cat.groupby('name_key')['sku'].agg(list)
multi = groups[groups.str.len() > 1]
for k, skus in multi.items():
    print(k, '->', skus)

24 beer light pack -> ['BEER-24', 'LTBEER24']
4 aa batteries pack -> ['BAT-AA4', 'BATT_AA_4PK']
aid first kit -> ['FAID-01', 'FIRSTAID-KIT']
board game monopoly -> ['BG-MONO', 'MONOPOLY-01', 'GAME-MONO']
bread white -> ['BREAD-WH', 'WHITEBREAD']
butter peanut -> ['PB-16', 'PNUTBTR-16']
candles emergency -> ['CNDL-EM', 'CANDLE-EMRG']
canned soup -> ['SOUP-CAN', 'CNSOUP-01']
cards playing -> ['CARDS-01', 'PLAYCARD-STD']
chips pack variety -> ['CHIPS-VP', 'CHIPVARIETY']
duty heavy tarp -> ['TARP-HD', 'HDTARP-01']
flashlight led -> ['FLASH-01', 'LED-FLASH']
paper towels -> ['PTOWEL-6', 'PAPERTOWEL6']
pop strawberry tarts -> ['PT-12', 'POPTART-STRAW', 'PT_STRAWBERRY']


The exact key misses near-misses (`tart` vs `tarts`, `24PK` vs `24-Pack`, `SZ 3` vs `Size 3`). A second, looser pass with light stemming and unit normalization surfaces them:

In [8]:

def loose_key(s):
    s = s.lower()
    s = re.sub(r'(\d+)\s*[- ]?\s*(pk|pack|roll)\b', r'\1pk', s)   # 24-pack / 24 pack / 24PK -> 24pk
    s = re.sub(r'\bsz\b', 'size', s)
    s = re.sub(r'[^a-z0-9\s]', ' ', s)
    toks = [t[:-1] if len(t) > 3 and t.endswith('s') else t for t in s.split()]   # tarts -> tart
    return ' '.join(sorted(toks))

cat['loose_key'] = cat['product_name'].map(loose_key)
g2 = cat.groupby('loose_key')['sku'].agg(list)
g2 = g2[g2.str.len() > 1]
new = g2[~g2.map(lambda l: any(l == v for v in multi.values))]
print(f"{len(g2)} groups with the loose key; {len(new)} are new or larger than before:\n")
for k, skus in new.items():
    print(f"{k!r:40} -> {skus}")


16 groups with the loose key; 4 are new or larger than before:

'12pk paper toilet'                      -> ['TP-12', 'TOILETPAPER12']
'24pk bottled water'                     -> ['WTR-24', 'WATER24PK']
'4pk aa batterie'                        -> ['BATT-100', 'BAT-AA4', 'BATT_AA_4PK']
'pop strawberry tart'                    -> ['PT-12', 'SKU#459812', 'POPTART-STRAW', 'PT_STRAWBERRY']


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [9]:
n_before = len(cat)
assert cat['canonical_sku'].notna().all(), "mapping produced nulls"
assert len(cat) == n_before, "row count changed"
print("OK:", n_before, "rows,", cat['canonical_sku'].nunique(), "distinct canonical SKUs")

OK: 45 rows, 42 distinct canonical SKUs


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

**a) Other fragmented product families**

- **Bottled water (3 SKUs):** `WTR-24`, `WATER24PK`, `WTR_24PK` ("BOTTLED WATER 24PK", "Bottled Water 24-Pack", "Water - 24 Pack"). The loose key (normalize `24PK`/`24-Pack`, ignore case) groups `WTR-24` + `WATER24PK` automatically, but `WTR_24PK` ("Water - 24 Pack", no word "bottled") slips through and needs a manual look. Canonical = `WTR-24`.
- **AA batteries (3 SKUs):** `BATT-100`, `BAT-AA4`, `BATT_AA_4PK`. The loose key groups all three (`aa` + `4pk` + `battery`); canonical = `BAT-AA4`.
- Also fragmented: **Monopoly** (3 SKUs, caught by the exact key), **toilet paper** (caught by the loose key), and **diapers size 3**, **infant/baby formula** and **portable charger**, which neither key catches and need manual or fuzzy matching. Same workflow: normalize, group, review by hand, then map.

**b) Where cleaning becomes a business decision**

Two examples where I'm not sure. (1) `STREAM-01` "Roku Streaming Stick" vs `ROKU-STD` "Streaming Device": the names don't match textually, and "Streaming Device" could be a different Roku model at a different price point. (2) `POPTART-STRAW` is a 6 pack while the other strawberry rows are 4-packs. Merging would hide a real difference in what is sold and in unit cost.

The point where it stops being cleanup is when a merge changes what the business counts as one product: pricing, demand forecasting, replenishment and supplier contracts all depend on it. I would ask the **category/merchandising manager** for Electronics and Breakfast (and the supplier or buyer who owns the item setup) whether these are the same sellable item before merging. Also note that `supplier`, `unit_cost` and `package_size` in this catalog are inconsistent even across rows that are clearly the same product, so I would not use them to decide a merge.